## French Embedding Fine-tuning (MTEB oriented)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

f:\codeAI\llm\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel


class F2LLMEmbedder(nn.Module):

    def __init__(self, model_name="codefuse-ai/F2LLM-v2-330M"):
        super().__init__()

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )

        self.model = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True
        )

    def encode(self, texts, device=None):

        if isinstance(texts, str):
            texts = [texts]

        if device is None:
            device = next(self.model.parameters()).device

        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = self.model(**batch)

        hidden = outputs.last_hidden_state

        eos_pos = batch["attention_mask"].sum(dim=1) - 1

        emb = hidden[
            torch.arange(hidden.size(0), device=device),
            eos_pos
        ]

        emb = F.normalize(emb, p=2, dim=1)

        return emb

    def forward(self, texts):

        batch = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(next(self.model.parameters()).device)

        outputs = self.model(**batch)

        hidden = outputs.last_hidden_state

        eos_pos = batch["attention_mask"].sum(dim=1) - 1

        emb = hidden[
            torch.arange(hidden.size(0), device=hidden.device),
            eos_pos
        ]

        emb = F.normalize(emb, p=2, dim=1)

        return emb

In [2]:
model = F2LLMEmbedder()

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
model.model.eval()

print("loaded")

f:\codeAI\llm\.venv\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aurel\.cache\huggingface\hub\models--codefuse-ai--F2LLM-v2-330M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 178/178 [00:00<00:00, 2710.48it/s]


loaded


In [3]:
emb = model.encode(["bonjour les ptits potes"])

print(emb.shape)

torch.Size([1, 896])


In [4]:
print(emb.norm(dim=1))

texts = [
    "Paris est la capitale de la France",
    "La capitale française est Paris",
    "Les chats aiment dormir",
]

emb = model.encode(texts)

sim = emb @ emb.T

print(sim.cpu())

tensor([1.], device='cuda:0', dtype=torch.bfloat16)
tensor([[1.0000, 0.9219, 0.1504],
        [0.9219, 0.9961, 0.1367],
        [0.1504, 0.1367, 1.0078]], dtype=torch.bfloat16)


In [5]:
import transformers
print(transformers.__version__)

5.12.1


In [6]:
texts = [
    "Paris est la capitale de la France",
    "La capitale française est Paris"
]

batch = model.tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
).to(device)

out = model.model(**batch)

loss = out.last_hidden_state.mean()

loss.backward()

print("grad ok")

grad ok


In [7]:
print(model.tokenizer.model_max_length)

131072


In [8]:
n_params = sum(p.numel() for p in model.parameters())

print(f"{n_params/1e6:.1f} M")

334.3 M


In [9]:
torch.cuda.empty_cache()

texts = ["bonjour"] * 64

emb = model.encode(texts)

print(torch.cuda.memory_allocated() / 1024**3)

1.2795929908752441


start

In [1]:
import torch
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

print("torch:", torch.__version__)

f:\codeAI\llm\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.5.1+cu121


In [25]:
model = F2LLMEmbedder(
    "codefuse-ai/F2LLM-v2-330M"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print(device)

Loading weights: 100%|██████████| 178/178 [00:00<00:00, 756.90it/s]


cuda


In [26]:
emb = model.encode(["bonjour"])
print(emb.shape)

torch.Size([1, 896])


In [13]:
from datasets import load_dataset

ds_100 = load_dataset(
    "sentence-transformers/miracl",
    "fr-triplet-100",
    split="train"
)

print(ds_100[0])
print(ds_100.column_names)

Generating train split: 100%|██████████| 2321/2321 [00:00<00:00, 3694.62 examples/s]


{'anchor': 'Qui était Princesse Diana?', 'positive': 'Diana Spencer\nLa « dianamania » bat son plein les premières années qui suivent la mort de la princesse de Galles, mais s\'essouffle progressivement et les lieux de pèlerinage sont moins fréquentés (place de l\'Alma, Althorp, dont est fermé en 2013). La célébration des vingt ans de sa mort, en 2017, suscitent à nouveau l\'engouement du public à son égard et provoquent une importante chute de popularité du prince Charles, critiqué pour sa façon de l\'avoir traitée.Dès son mariage et jusqu\'à son divorce, sa titulature était : "Son Altesse Royale la princesse de Galles et comtesse de Chester, duchesse de Cornouailles, duchesse de Rothesay, comtesse de Carrick, baronne de Renfrew, Dame des Îles, princesse d\'Écosse" ; mais elle portait comme titre officiel "Son Altesse Royale la princesse de Galles". Après son divorce en 1996, la reine lui accorde de conserver le titre de princesse de Galles comme titre de courtoisie sans prédicat. Ell

In [14]:
row = ds_100[0]

anchor = row["anchor"]
positive = row["positive"]
negative = row["negative_1"]

print(anchor[:200])
print()
print(positive[:200])
print()
print(negative[:200])

Qui était Princesse Diana?

Diana Spencer
La « dianamania » bat son plein les premières années qui suivent la mort de la princesse de Galles, mais s'essouffle progressivement et les lieux de pèlerinage sont moins fréquentés (pla

Diana : À la recherche de la vérité
Aidée par un inspecteur de police, une journaliste enquête sur la mort tragique de la princesse Diana.


In [16]:
from sentence_transformers import InputExample

train_examples = []

for row in ds_100:

    train_examples.append(
        InputExample(
            texts=[
                row["anchor"],
                row["positive"],
                row["negative_1"]
            ]
        )
    )

print("triplets:", len(train_examples))
print(train_examples[0].texts[0][:100])

triplets: 2321
Qui était Princesse Diana?


In [18]:
from torch.utils.data import Dataset

class TripletDataset(Dataset):

    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):

        anchor, positive, negative = self.examples[idx].texts

        return {
            "anchor": anchor,
            "positive": positive,
            "negative": negative
        }


train_dataset = TripletDataset(train_examples)

print(len(train_dataset))
print(train_dataset[0])

2321
{'anchor': 'Qui était Princesse Diana?', 'positive': 'Diana Spencer\nLa « dianamania » bat son plein les premières années qui suivent la mort de la princesse de Galles, mais s\'essouffle progressivement et les lieux de pèlerinage sont moins fréquentés (place de l\'Alma, Althorp, dont est fermé en 2013). La célébration des vingt ans de sa mort, en 2017, suscitent à nouveau l\'engouement du public à son égard et provoquent une importante chute de popularité du prince Charles, critiqué pour sa façon de l\'avoir traitée.Dès son mariage et jusqu\'à son divorce, sa titulature était : "Son Altesse Royale la princesse de Galles et comtesse de Chester, duchesse de Cornouailles, duchesse de Rothesay, comtesse de Carrick, baronne de Renfrew, Dame des Îles, princesse d\'Écosse" ; mais elle portait comme titre officiel "Son Altesse Royale la princesse de Galles". Après son divorce en 1996, la reine lui accorde de conserver le titre de princesse de Galles comme titre de courtoisie sans prédicat

In [20]:
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

print(len(train_loader))

291


In [21]:
batch = next(iter(train_loader))

print(batch["anchor"][0][:100])
print()
print(batch["positive"][0][:100])
print()
print(batch["negative"][0][:100])

Quelle est la meilleure école d'arts au Canada?

École nationale de théâtre du Canada
L'École nationale de théâtre est aujourd'hui considérée comme l

Artisans dans la société canadienne sous le régime français
L'École des arts et métiers de Saint-Joa


In [27]:
a = train_dataset[0]["anchor"]

emb = model.forward([a])

print(emb.shape)
print(emb.requires_grad)

torch.Size([1, 896])
True


In [28]:
criterion = torch.nn.TripletMarginLoss(
    margin=0.3,
    p=2
)

criterion

TripletMarginLoss()

In [29]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

In [30]:
batch = next(iter(train_loader))

anchor_emb = model.forward(batch["anchor"])
positive_emb = model.forward(batch["positive"])
negative_emb = model.forward(batch["negative"])

loss = criterion(
    anchor_emb,
    positive_emb,
    negative_emb
)

print(loss.item())

0.193359375
